# GSM8K agent — starter notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ml-arena/competition-baseline/blob/main/gsm8k/agent_baseline.ipynb)

A minimal, runnable baseline for the **GSM8K** math-reasoning competition
([#165](https://ml-arena.com/viewcompetition/165)). The environment sends your agent
batches of **10 word problems** and expects one numeric answer per problem.

> This template just **guesses random numbers** so you can see the full submit loop.
> The real work is the `answer` method — swap in a small language model (see below).

## 0. Setup

In [ ]:
!pip install -q "mlarena-sdk==0.3.0" numpy   # installs the `mlarena` package

import mlarena

API_TOKEN = "mlk_user_REPLACE_ME"   # <-- paste your token (Profile page -> API Keys)
COMPETITION_ID = 165

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. Define your agent

The env calls `answer(questions: list[str])` once per subset of 10 and expects a tuple
`(solutions: list[float], thinking_traces: list[str])` — one entry per question, in
order. A reply is correct when `abs(solution - gold) < 1e-6`. Each batch has a
**60-second timeout**. `__init__` takes no arguments.

In [ ]:
import random


class Agent:
    def __init__(self):
        self.rng = random.Random(0)

    def answer(self, questions):
        solutions, traces = [], []
        for q in questions:
            sol, trace = self._solve_one(q)
            solutions.append(sol)
            traces.append(trace)
        return solutions, traces

    def _solve_one(self, question):
        # TODO: replace with real reasoning (see cell below).
        guess = self.rng.uniform(0, 10)
        return guess, f'random guess: {guess:.4f}'

## 2. (optional) Try it locally

In [ ]:
agent = Agent()
sols, traces = agent.answer(['Natalia sold 48 clips, then half as many. Total?'])
print(sols, traces)

## 3. Submit

In [ ]:
# Uploads the Agent class source as `agent.py`, then creates + deploys the attachment.
result = client.submit(competition_id=COMPETITION_ID, agent=Agent)
print(result)

# Poll until the run reaches a terminal state (deploy -> queue -> run -> scored).
for line in client.tail_logs(COMPETITION_ID, result["attache_agent_id"]):
    print(line)

## 4. Leaderboard

In [ ]:
client.leaderboard(COMPETITION_ID)

## 5. Where to go from here — use a real model

The scoring baseline is a **small cached language model** with chain-of-thought. The
evaluation image pre-caches ~1–2B instruct models (e.g. `Qwen/Qwen2.5-0.5B-Instruct`,
`Qwen/Qwen3-1.7B`, `HuggingFaceTB/SmolLM2-1.7B-Instruct`) so you don't spend your
60s budget downloading. A minimal strategy:

1. In `__init__`, load a cached model:
   `AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")` (+ tokenizer).
2. In `answer`, prompt it to reason step by step and end with `#### <number>`, then
   regex the final number out of each reply. Batch the 10 questions in one
   `model.generate(...)` call to stay inside the timeout.
3. Submit on a GPU runtime:
   `client.submit(COMPETITION_ID, agent=Agent, runtime={"language": "python", "framework": "torch"})`.

Learn more: HF LLM course <https://huggingface.co/learn/llm-course/chapter1/1> and
the Agents course <https://huggingface.co/learn/agents-course/unit0/introduction>.